In [23]:
from pathlib import Path
import os
import pandas as pd
import requests
from decimal import Decimal

from nautilus_trader.model.identifiers import InstrumentId, Symbol
from nautilus_trader.model.identifiers import InstrumentId, Symbol
from nautilus_trader.model.instruments import Instrument, CryptoPerpetual
from nautilus_trader.model.objects import Currency, Price, Quantity, Money

from nautilus_trader.persistence.catalog import ParquetDataCatalog

from nautilus_trader.adapters.bybit import BybitProductType

from nautilus_trader.model.data import FundingRateUpdate

In [24]:
SYMBOL = "ETHUSDT"
EXCHANGE = "BYBIT"
INSTRUMENT_ID = f"{SYMBOL}-LINEAR.{EXCHANGE}"
instrument_id = InstrumentId.from_str(INSTRUMENT_ID)
DATA_DIR = Path(os.environ.get("DATA_DIR", "~/desktop/tmpMarketData/FundingRate")).expanduser() / SYMBOL
CATALOG_DIR = Path(os.getcwd()).parent/"nautilusDataCatalog"

In [25]:
df = pd.read_csv(
    DATA_DIR / "ETHUSDT_funding_rate_2021-01-01_to_2026-06-29.csv",
    engine="pyarrow",
    dtype={
        "symbol": "string",
        "funding_rate": "string",
        "funding_rate_timestamp_ms": "int64",
    },
    )

df

,symbol,funding_rate_timestamp_ms,funding_rate_time_utc,funding_rate
0,ETHUSDT,1609459200000,2021-01-01 00:00:00+00:00,0.00040403
1,ETHUSDT,1609488000000,2021-01-01 08:00:00+00:00,0.00031725
2,ETHUSDT,1609516800000,2021-01-01 16:00:00+00:00,0.0005071
3,ETHUSDT,1609545600000,2021-01-02 00:00:00+00:00,0.00051118
4,ETHUSDT,1609574400000,2021-01-02 08:00:00+00:00,0.00034563
...,...,...,...,...
6013,ETHUSDT,1782633600000,2026-06-28 08:00:00+00:00,-4.872e-05
6014,ETHUSDT,1782662400000,2026-06-28 16:00:00+00:00,2.49e-06
6015,ETHUSDT,1782691200000,2026-06-29 00:00:00+00:00,-2.896e-05
6016,ETHUSDT,1782720000000,2026-06-29 08:00:00+00:00,3.363e-05


In [26]:
# Convert Bybit milliseconds to Nautilus nanoseconds
df["ts_event"] = (
    pd.to_numeric(
        df["funding_rate_timestamp_ms"],
        errors="raise",
    ).astype("int64")
    * 1_000_000
)

# Remove bad or duplicate rows and sort chronologically
df = (
    df.dropna(subset=["funding_rate", "ts_event"])
    .sort_values("ts_event", kind="stable")
    .drop_duplicates(subset=["ts_event"], keep="last")
    .reset_index(drop=True)
)

In [27]:
funding_updates = [
    FundingRateUpdate(
        instrument_id=instrument_id,
        rate=Decimal(str(row.funding_rate)),
        ts_event=int(row.ts_event),
        ts_init=int(row.ts_event),
        interval=480,          # 8 hours
        next_funding_ns=None,
    )
    for row in df.itertuples(index=False)
]
print(type(funding_updates))
print(type(funding_updates[0]))
print(funding_updates[0])
print(f"Created {len(funding_updates):,} funding-rate updates")

<class 'list'>
<class 'nautilus_trader.model.data.FundingRateUpdate'>
FundingRateUpdate(instrument_id=ETHUSDT-LINEAR.BYBIT, rate=0.00040403, interval=480, next_funding_ns=None, ts_event=1609459200000000000, ts_init=1609459200000000000)
Created 6,018 funding-rate updates


In [28]:
# Get instrument specs from Bybit API

def get_bybit_linear_instrument_info(symbol: str, testnet: bool = False) -> dict:
    base_url = "https://api-testnet.bybit.com" if testnet else "https://api.bybit.com"

    params = {
        "category": "linear",
        "symbol": symbol.upper(),
    }

    r = requests.get(
        f"{base_url}/v5/market/instruments-info",
        params=params,
        timeout=20,
    )
    r.raise_for_status()

    payload = r.json()

    if payload["retCode"] != 0:
        raise RuntimeError(payload)

    instruments = payload["result"]["list"]

    if not instruments:
        raise ValueError(f"No Bybit linear instrument found for {symbol}")

    return instruments[0]

info = get_bybit_linear_instrument_info(SYMBOL)

In [29]:
symbol = info["symbol"]              
base_coin = info["baseCoin"]         
quote_coin = info["quoteCoin"]      
settle_coin = info["settleCoin"]     

tick_size = info["priceFilter"]["tickSize"]
qty_step = info["lotSizeFilter"]["qtyStep"]

price_precision = int(info["priceScale"])
size_precision = abs(Decimal(qty_step).as_tuple().exponent)

min_qty = info["lotSizeFilter"]["minOrderQty"]
max_qty = info["lotSizeFilter"]["maxOrderQty"]
min_notional = info["lotSizeFilter"]["minNotionalValue"]

min_price = info["priceFilter"]["minPrice"]
max_price = info["priceFilter"]["maxPrice"]

max_leverage = Decimal(info["leverageFilter"]["maxLeverage"])
margin_init = Decimal("1") / max_leverage

In [30]:
CRYPTOPERP_INSTRUMENT = CryptoPerpetual(
    instrument_id=InstrumentId.from_str(f"{symbol}-LINEAR.BYBIT"),
    raw_symbol=Symbol(symbol),

    base_currency=Currency.from_str(base_coin),
    quote_currency=Currency.from_str(quote_coin),
    settlement_currency=Currency.from_str(settle_coin),

    is_inverse=False,

    price_precision=price_precision,
    size_precision=size_precision,

    price_increment=Price.from_str(tick_size),
    size_increment=Quantity.from_str(qty_step),

    multiplier=Quantity.from_str("1"),
    lot_size=Quantity.from_str("1"),

    min_quantity=Quantity.from_str(min_qty),
    max_quantity=Quantity.from_str(max_qty),

    min_notional=Money.from_str(f"{min_notional} {quote_coin}"),
    max_notional=None,

    min_price=Price.from_str(min_price),
    max_price=Price.from_str(max_price),

    margin_init=margin_init,
    margin_maint=Decimal("0"),

    maker_fee=Decimal("0.0002"),
    taker_fee=Decimal("0.00055"),

    ts_event=0,
    ts_init=0,

    info=info,
)

CRYPTOPERP_INSTRUMENT

CryptoPerpetual(id=ETHUSDT-LINEAR.BYBIT, raw_symbol=ETHUSDT, asset_class=CRYPTOCURRENCY, instrument_class=SWAP, quote_currency=USDT, is_inverse=False, price_precision=2, price_increment=0.01, size_precision=2, size_increment=0.01, multiplier=1, lot_size=1, margin_init=0.01, margin_maint=0, maker_fee=0.0002, taker_fee=0.00055, info={'symbol': 'ETHUSDT', 'contractType': 'LinearPerpetual', 'status': 'Trading', 'baseCoin': 'ETH', 'quoteCoin': 'USDT', 'launchTime': '1615766400000', 'deliveryTime': '0', 'deliveryFeeRate': '', 'priceScale': '2', 'leverageFilter': {'minLeverage': '1', 'maxLeverage': '100.00', 'leverageStep': '0.01'}, 'priceFilter': {'minPrice': '0.01', 'maxPrice': '199999.98', 'tickSize': '0.01'}, 'lotSizeFilter': {'maxOrderQty': '10000.00', 'minOrderQty': '0.01', 'qtyStep': '0.01', 'postOnlyMaxOrderQty': '10000.00', 'maxMktOrderQty': '2000.00', 'minNotionalValue': '5'}, 'unifiedMarginTrade': True, 'fundingInterval': 480, 'settleCoin': 'USDT', 'copyTrading': 'both', 'upperFund

In [31]:
catalog = ParquetDataCatalog(str(CATALOG_DIR))
catalog.write_data([CRYPTOPERP_INSTRUMENT])

catalog.write_data(funding_updates)

File /Users/damensavvasavvi/Desktop/NautilusTrader/project/nautilusDataCatalog/data/crypto_perpetual/ETHUSDT-LINEAR.BYBIT/1970-01-01T00-00-00-000000000Z_1970-01-01T00-00-00-000000000Z.parquet already exists, skipping write
